In [5]:
from pathlib import Path
from typing import List, Optional, Tuple, Set, Dict
import io
import fitz  # PyMuPDF
import cv2
import numpy as np
from math import atan2, degrees, hypot, ceil
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import os
import uuid

try:
    from PIL import Image  # only needed for JPEG encoding
    _HAS_PIL = True
except Exception:
    _HAS_PIL = False


In [6]:
def read_file_as_binary(file_path):
    """
    Reads a local file and returns its binary content.
    
    Args:
        file_path (str): Path to the file.
    
    Returns:
        bytes: Binary content of the file.
    """
    with open(file_path, "rb") as f:
        binary_data = f.read()
    return binary_data


In [7]:
def pdf_bytes_to_images(
    pdf_bytes: bytes,
    dpi: int = 200,
    fmt: str = "png",                   # "png" or "jpg"
    save_dir: Optional[Path | str] = None,
    base_filename: Optional[str] = None,  # used for saved filenames only
    jpg_quality: int = 90
) -> List[bytes]:
    """
    Render each page of a (scanned) PDF (given as bytes) to images and return a list of image bytes.
    Optionally save images to disk as <base_filename>_p<page>.<ext> (1-based page numbering).

    Args:
        pdf_bytes: Raw bytes of the PDF.
        dpi: Output DPI (controls resolution). 72 dpi == 1.0 zoom.
        fmt: "png" (native via PyMuPDF) or "jpg".
        save_dir: If provided, images are also saved to this directory.
        base_filename: If saving, the basename to use (default: "document").
        jpg_quality: JPEG quality (1–95) if fmt="jpg".

    Returns:
        List of bytes objects, one per page, in the requested format.
    """
    fmt = fmt.lower()
    if fmt not in {"png", "jpg", "jpeg"}:
        raise ValueError("fmt must be 'png' or 'jpg'")

    if fmt in {"jpg", "jpeg"} and not _HAS_PIL:
        raise RuntimeError("JPEG output requires Pillow (pip install Pillow)")

    # Prepare output directory (optional)
    out_dir: Optional[Path] = None
    if save_dir is not None:
        out_dir = Path(save_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        if not base_filename:
            base_filename = "document"

    if not base_filename:
        base_filename = "document"

    # DPI -> zoom factor
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)

    images: List[bytes] = []

    # Open from bytes
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        if doc.needs_pass:
            raise RuntimeError("This PDF is encrypted and needs a password.")
        page_count = doc.page_count

        for page_index, page in enumerate(doc, start=1):
            # Render to pixmap (no alpha for cleaner JPG/PNG)
            pix = page.get_pixmap(matrix=mat, alpha=False)

            if fmt == "png":
                img_bytes = pix.tobytes(output="png")  # PyMuPDF-native PNG
                ext = "png"
            else:
                # Convert Pixmap -> PIL Image -> JPEG bytes
                # pix.samples are RGB bytes; pix.stride is bytes per row
                mode = "RGB" if pix.n in (3, 4) else "L"
                pil_img = Image.frombytes(mode, (pix.width, pix.height), pix.samples)
                buf = io.BytesIO()
                pil_img.save(buf, format="JPEG", quality=jpg_quality, optimize=True)
                img_bytes = buf.getvalue()
                ext = "jpg"

            images.append(img_bytes)

            # Optional save
            if out_dir is not None:
                filename = f"{base_filename}_p{page_index}.{ext}"
                (out_dir / filename).write_bytes(img_bytes)

    return images


In [27]:
# pip install -U langchain langchain-google-genai google-generativeai pydantic
import os
import json
import re
import base64
from typing import Any, Dict, List, Union

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.messages import SystemMessage, HumanMessage

from app.core.config import settings
import time


# ----------------------------
# CONFIG
# ----------------------------
API_KEY = settings.api_key

# Choose a Gemini model. For best quality/format reliability, use 1.5-pro at low temperature.
MODEL_NAME = settings.chatbot_model
TEMPERATURE = 1.0


# ----------------------------
# UTIL: load instruction text
# ----------------------------
def load_instruction_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


# ----------------------------
# UTIL: strip code fences and parse JSON
# ----------------------------
def extract_json(text: str) -> Dict[str, Any]:
    """
    Extract JSON from a string that may contain markdown fences.
    """
    # Grab content between ```json ... ``` or plain {...}
    fence_match = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.S)
    if fence_match:
        payload = fence_match.group(1)
    else:
        # If no fenced block, try to find the first {...} JSON object
        brace_match = re.search(r"(\{.*\})", text, flags=re.S)
        if not brace_match:
            raise ValueError("No JSON object found in the model output.")
        payload = brace_match.group(1)

    # Parse
    return json.loads(payload)

def _image_to_data_url(img: Union[str, bytes]) -> str:
    """
    Convert an image (file path or raw bytes) to a data URL for Gemini.

    Supports PNG / JPG / JPEG by extension. Defaults to image/png if unknown.
    """
    if isinstance(img, str):
        # treat as file path
        with open(img, "rb") as f:
            data = f.read()
        ext = os.path.splitext(img)[1].lower()
        if ext in [".jpg", ".jpeg"]:
            mime = "image/jpeg"
        else:
            mime = "image/png"
    else:
        # assume raw bytes
        data = img
        mime = "image/png"

    b64 = base64.b64encode(data).decode("ascii")
    return f"data:{mime};base64,{b64}"


# global rate-limit state (shared across all calls)
_rate_state = {
    "window_start": 0.0,
    "calls_in_window": 0,
}


def _apply_rate_limit(window_seconds: int, max_calls_per_window: int) -> None:
    """
    Enforce a simple sliding-window rate limit shared across all
    calls to transform_images_to_json.

    At most `max_calls_per_window` calls in each `window_seconds` window.
    """
    global _rate_state

    now = time.time()
    window_start = _rate_state["window_start"]
    calls_in_window = _rate_state["calls_in_window"]

    # If first call ever or window has expired, reset the window
    if window_start == 0.0 or (now - window_start) >= window_seconds:
        window_start = now
        calls_in_window = 0

    # If we've hit the limit, sleep until the window resets
    if calls_in_window >= max_calls_per_window:
        sleep_for = window_seconds - (now - window_start)
        if sleep_for > 0:
            print(f"[Rate limit] Reached {max_calls_per_window} calls in "
                  f"{window_seconds}s, sleeping for {sleep_for:.1f} seconds...")
            time.sleep(sleep_for)
            print("[Rate limit] Resuming after sleep.")
        # start a new window after sleeping
        window_start = time.time()
        calls_in_window = 0

    # record this call
    calls_in_window += 1
    _rate_state["window_start"] = window_start
    _rate_state["calls_in_window"] = calls_in_window

def transform_images_to_json(
    images: List[Union[str, bytes]],
    instruction_path: str = "image_ocr_instructions.txt",
    WINDOW_SECONDS: int = 60,
    MAX_CALLS_PER_WINDOW: int = 15,
) -> Dict[str, Any]:
    """
    Send one or more images to Gemini with OCR-style instructions
    and get back JSON of the form:
        {
          "result": [...],
          "_usage": {
              "input_tokens": int | None,
              "output_tokens": int | None,
              "total_tokens": int | None
          }
        }

    - images: list of file paths or raw image bytes.
    - instruction_path: path to the instruction .txt you wrote.
    """
    if not images:
        return {"result": [], "_usage": None}

    instruction_text = load_instruction_text(instruction_path).strip()

    llm = ChatGoogleGenerativeAI(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        api_key=API_KEY,
    )

    # ---- rate limiting disabled for parallel calls ----
    # _apply_rate_limit(WINDOW_SECONDS, MAX_CALLS_PER_WINDOW)

    parts: List[dict] = []

    parts.append({
        "type": "text",
        "text": (
            "You will now receive several images. For each image, extract text "
            "according to the instructions and then return one JSON object at the end."
        ),
    })

    for idx, img in enumerate(images, start=1):
        data_url = _image_to_data_url(img)
        parts.append({"type": "text", "text": f"Image {idx}:"})
        parts.append({"type": "image_url", "image_url": data_url})

    system_msg = SystemMessage(
        content=instruction_text + "\n\nImportant: Return ONLY the JSON object."
    )
    human_msg = HumanMessage(content=parts)

    response = llm.invoke([system_msg, human_msg])

    raw_text = (response.content or "").strip()
    parsed = extract_json(raw_text)

    # Get usage metadata from the AIMessage (if available)
    usage = getattr(response, "usage_metadata", None)

    # Attach it into the returned JSON without breaking existing code
    if isinstance(parsed, dict):
        parsed["_usage"] = usage
    else:
        # If your instructions ever return a non-dict, wrap it
        parsed = {"result": parsed, "_usage": usage}

    return parsed

In [10]:
# Bounding box type: (x_min, y_min, x_max, y_max)
BBox = Tuple[int, int, int, int]


def crop_image_array(img: np.ndarray, bbox: BBox, margin=0) -> np.ndarray:
    """
    Crop an image (NumPy array) using pixel coordinates.

    Args:
        img: HxWxC or HxW image (BGR if from cv2).
        bbox: (x_min, y_min, x_max, y_max) in *pixel coordinates*.

    Returns:
        Cropped image as NumPy array.
    """
    if img is None:
        raise ValueError("crop_image_array: img is None")

    h, w = img.shape[:2]
    x_min, y_min, x_max, y_max = bbox

    # Clamp to image bounds
    x_min = int(max(0, min(x_min-margin, w)))
    x_max = int(max(0, min(x_max+margin, w)))
    y_min = int(max(0, min(y_min-margin, h)))
    y_max = int(max(0, min(y_max+margin, h)))

    if x_max <= x_min or y_max <= y_min:
        raise ValueError(f"Invalid bbox after clamping: {bbox}")

    # NumPy slices: [y_min:y_max, x_min:x_max]
    crop = img[y_min:y_max, x_min:x_max].copy()
    return crop


def crop_image_file(
    input_path: str,
    bbox: BBox,
    margin=0,
    output_path: str = None,
) -> np.ndarray:
    """
    Load an image from disk, crop by pixel bbox, optionally save.

    Args:
        input_path: path to input image.
        bbox: (x_min, y_min, x_max, y_max) in pixels.
        output_path: where to save cropped image (if not None).

    Returns:
        Cropped image as NumPy array.
    """
    img = cv2.imread(input_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {input_path}")

    crop = crop_image_array(img, bbox, margin=margin)

    if output_path is not None:
        cv2.imwrite(output_path, crop)

    return crop

def image_array_to_png_bytes(img: np.ndarray) -> bytes:
    """
    Encode an OpenCV image (np.ndarray, BGR) as PNG bytes.
    """
    ok, buf = cv2.imencode(".png", img)
    if not ok:
        raise ValueError("Failed to encode image to PNG")
    return buf.tobytes()

In [9]:
PATH = "Data input.pdf"
typ = "pdf"

file = read_file_as_binary(PATH)

imgs = pdf_bytes_to_images(
    file,
    dpi=400,
    fmt="png",
    save_dir="testing",  # or None to skip saving
    base_filename="my_scan"           # filename prefix for saved files
)

print(f"Returned {len(imgs)} images as bytes.")

Returned 13 images as bytes.


In [22]:
img_path = "testing/my_scan_p4.png"
img = cv2.imread(img_path)

bbox = [0, 0, img.shape[1], img.shape[0] /2]
# bbox = [0, img.shape[0] /2, img.shape[1], img.shape[0]]

# From file
crop = crop_image_file(img_path, bbox, margin=1,output_path="crop_test.png")

In [50]:
bits = []
img_paths = ["testing/my_scan_p3.png"]
for img_path in img_paths:
    bit = read_file_as_binary(img_path)
    bits.append(bit)

result = transform_images_to_json(images=bits, instruction_path="instruction.txt")

In [51]:
result

{'result': [{'y': 'Y2', 'x': [129, 187, 89, 34, 40, 41, 44, 45, 76, 34]},
  {'y': 'Y3', 'x': [6.9, None, 99, 33, None, None, None, 32, 25, None]}],
 '_usage': {'input_tokens': 878,
  'output_tokens': 161,
  'total_tokens': 2068,
  'input_token_details': {'cache_read': 0}}}